# Replay Data from MinIO to Kafka

This notebook allows on-demand loading of Parquet files from MinIO back into Kafka.
Target topic: `order-events-replay` (separate from live data)

In [1]:
# Cell 1: Imports and Configuration
import s3fs
import pyarrow.parquet as pq
import pandas as pd
from collections import defaultdict
from confluent_kafka import Producer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import SerializationContext, MessageField

# Configuration
KAFKA_BOOTSTRAP = "broker-i1:19092,broker-i2:19092,broker-i3:19092"
SCHEMA_REGISTRY = "http://schema-registry:8081"
TARGET_TOPIC = "order-events-replay"
MINIO_BUCKET = "datalake"
SOURCE_TOPIC_PATH = "topics/order-events"

# S3/MinIO configuration
s3 = s3fs.S3FileSystem(
    key='minioadmin',
    secret='minioadmin',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

print("Configuration loaded successfully")
print(f"Target Kafka topic: {TARGET_TOPIC}")
print(f"Schema Registry: {SCHEMA_REGISTRY}")

Configuration loaded successfully
Target Kafka topic: order-events-replay
Schema Registry: http://schema-registry:8081


In [2]:
# Cell 2: Show available partitions in MinIO
s3.invalidate_cache()

# Find all parquet files
base_path = f"{MINIO_BUCKET}/{SOURCE_TOPIC_PATH}"
all_parquet_files = s3.glob(f"{base_path}/**/*.parquet")

print(f"Found {len(all_parquet_files)} Parquet files\n")

# Group by partition (calc_id/dt/hour)
partitions = defaultdict(list)

for file_path in all_parquet_files:
    path_parts = file_path.split('/')
    # Extract calc_id, dt, hour from path
    calc_id = dt = hour = None
    for part in path_parts:
        if part.startswith('calc_id='):
            calc_id = part
        elif part.startswith('dt='):
            dt = part
        elif part.startswith('hour='):
            hour = part
    
    if calc_id and dt and hour:
        partition_key = f"{calc_id}/{dt}/{hour}"
        partitions[partition_key].append(file_path)

# Show statistics for each partition
print("Available partitions:")
print("=" * 80)

partition_stats = []
for partition_key, files_list in sorted(partitions.items()):
    total_rows = 0
    total_size = 0
    for file_path in files_list:
        try:
            with s3.open(file_path, 'rb') as f:
                pf = pq.ParquetFile(f)
                total_rows += pf.metadata.num_rows
            info = s3.info(file_path)
            total_size += info.get('Size', 0)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue
    
    partition_stats.append({
        'partition': partition_key,
        'files': len(files_list),
        'rows': total_rows,
        'size_mb': total_size / 1024 / 1024
    })
    print(f"{partition_key}")
    print(f"  Files: {len(files_list)}, Rows: {total_rows:,}, Size: {total_size / 1024 / 1024:.2f} MB")

print("\n" + "=" * 80)
total_files = sum(p['files'] for p in partition_stats)
total_rows = sum(p['rows'] for p in partition_stats)
total_size = sum(p['size_mb'] for p in partition_stats)
print(f"TOTAL: {total_files} files, {total_rows:,} rows, {total_size:.2f} MB")

Found 144 Parquet files

Available partitions:
calc_id=20251224-180000/dt=2025-12-24/hour=18
  Files: 144, Rows: 2,000,000, Size: 14.25 MB

TOTAL: 144 files, 2,000,000 rows, 14.25 MB


In [3]:
# Cell 3: Parameters for replay (specify the partition to load)
# Copy one of the partition paths from the cell above

PARTITION_PATH = "calc_id=20251224-180000/dt=2025-12-24/hour=18"

# Full path in MinIO
full_partition_path = f"{MINIO_BUCKET}/{SOURCE_TOPIC_PATH}/{PARTITION_PATH}"
print(f"Selected partition: {PARTITION_PATH}")
print(f"Full path: {full_partition_path}")

# Verify partition exists
partition_files = s3.glob(f"{full_partition_path}/*.parquet")
print(f"\nFiles in partition: {len(partition_files)}")

if partition_files:
    total_rows = 0
    for file_path in partition_files:
        with s3.open(file_path, 'rb') as f:
            pf = pq.ParquetFile(f)
            total_rows += pf.metadata.num_rows
    print(f"Total rows to replay: {total_rows:,}")
else:
    print("WARNING: No files found in this partition!")

Selected partition: calc_id=20251224-180000/dt=2025-12-24/hour=18
Full path: datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18

Files in partition: 144
Total rows to replay: 2,000,000


In [4]:
# Cell 4: Setup Kafka Producer with Avro serialization

# Get schema from Schema Registry
schema_registry_client = SchemaRegistryClient({'url': SCHEMA_REGISTRY})

# Get the latest schema for OrderEvent
# The subject name follows the TopicNameStrategy: {topic}-value
SCHEMA_SUBJECT = "order-events-value"

try:
    schema = schema_registry_client.get_latest_version(SCHEMA_SUBJECT)
    avro_schema_str = schema.schema.schema_str
    print(f"Schema subject: {SCHEMA_SUBJECT}")
    print(f"Schema version: {schema.version}")
    print(f"Schema ID: {schema.schema_id}")
except Exception as e:
    print(f"Error getting schema: {e}")
    print("\nTrying alternative subject name...")
    # Try with full class name
    SCHEMA_SUBJECT = "ru.pospelov.etl.avro.OrderEvent"
    schema = schema_registry_client.get_latest_version(SCHEMA_SUBJECT)
    avro_schema_str = schema.schema.schema_str
    print(f"Found schema: {SCHEMA_SUBJECT}")

Schema subject: order-events-value
Schema version: 2
Schema ID: 2


In [5]:
# Cell 5: Create Avro serializer and Kafka producer

def record_to_dict(record, ctx):
    """Convert record to dictionary for Avro serialization."""
    return record

# Create Avro serializer
avro_serializer = AvroSerializer(
    schema_registry_client,
    avro_schema_str,
    record_to_dict
)

# Kafka producer configuration
producer_config = {
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'linger.ms': 100,
    'batch.size': 65536,
    'acks': 'all',
    'compression.type': 'snappy',
    'retries': 3,
    'retry.backoff.ms': 500
}

producer = Producer(producer_config)
print("Kafka producer created successfully")
print(f"Target topic: {TARGET_TOPIC}")

Kafka producer created successfully
Target topic: order-events-replay


In [6]:
# Cell 6: Read Parquet and send to Kafka

# Counters for tracking
sent_count = 0
error_count = 0
BATCH_SIZE = 1000
PROGRESS_INTERVAL = 10000

def delivery_callback(err, msg):
    """Callback for delivery reports."""
    global sent_count, error_count
    if err:
        error_count += 1
        if error_count <= 10:  # Only print first 10 errors
            print(f"Delivery error: {err}")
    else:
        sent_count += 1

def convert_row_to_avro(row):
    """Convert DataFrame row to Avro-compatible dictionary."""
    record = {}
    for col, value in row.items():
        # Skip partition columns (they're metadata, not part of original event)
        if col in ('calc_id', 'dt', 'hour'):
            continue
        
        # Handle None/NaN values
        if pd.isna(value):
            record[col] = None
        # Handle numpy types
        elif hasattr(value, 'item'):
            record[col] = value.item()
        else:
            record[col] = value
    
    return record

# Get list of files to process
parquet_files = s3.glob(f"{full_partition_path}/*.parquet")
print(f"Processing {len(parquet_files)} files from partition: {PARTITION_PATH}")
print(f"Sending to topic: {TARGET_TOPIC}")
print("=" * 60)

total_rows_processed = 0
serialization_context = SerializationContext(TARGET_TOPIC, MessageField.VALUE)

for i, file_path in enumerate(parquet_files):
    try:
        # Read parquet file
        df = pd.read_parquet(f"s3://{file_path}", filesystem=s3)
        file_rows = len(df)
        
        # Process each row
        for idx, row in df.iterrows():
            # Convert row to Avro record
            record = convert_row_to_avro(row)
            
            # Serialize with Avro
            try:
                value_bytes = avro_serializer(record, serialization_context)
                
                # Use order_id as key for partitioning
                key = record.get('order_id', '').encode('utf-8') if record.get('order_id') else None
                
                # Send to Kafka
                producer.produce(
                    topic=TARGET_TOPIC,
                    key=key,
                    value=value_bytes,
                    callback=delivery_callback
                )
            except Exception as e:
                error_count += 1
                if error_count <= 10:
                    print(f"Serialization error: {e}")
                continue
            
            total_rows_processed += 1
            
            # Poll for callbacks every BATCH_SIZE records
            if total_rows_processed % BATCH_SIZE == 0:
                producer.poll(0)
            
            # Progress report
            if total_rows_processed % PROGRESS_INTERVAL == 0:
                print(f"Progress: {total_rows_processed:,} rows processed, {sent_count:,} sent, {error_count} errors")
        
        print(f"File {i+1}/{len(parquet_files)}: {file_path.split('/')[-1]} - {file_rows:,} rows")
        
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        continue

# Flush remaining messages
print("\nFlushing remaining messages...")
producer.flush(timeout=60)

print("\n" + "=" * 60)
print(f"REPLAY COMPLETED")
print(f"Total rows processed: {total_rows_processed:,}")
print(f"Successfully sent: {sent_count:,}")
print(f"Errors: {error_count}")
print(f"Target topic: {TARGET_TOPIC}")

Processing 144 files from partition: calc_id=20251224-180000/dt=2025-12-24/hour=18
Sending to topic: order-events-replay
Progress: 10,000 rows processed, 0 sent, 0 errors
File 1/144: order-events+0+0000000000.snappy.parquet - 13,976 rows
Progress: 20,000 rows processed, 19,566 sent, 0 errors
File 2/144: order-events+0+0000013976.snappy.parquet - 13,976 rows
Progress: 30,000 rows processed, 29,568 sent, 0 errors
Progress: 40,000 rows processed, 39,571 sent, 0 errors
File 3/144: order-events+1+0000000000.snappy.parquet - 13,745 rows
Progress: 50,000 rows processed, 49,574 sent, 0 errors
File 4/144: order-events+1+0000013745.snappy.parquet - 13,745 rows
Progress: 60,000 rows processed, 59,298 sent, 0 errors
File 5/144: order-events+10+0000000000.snappy.parquet - 13,728 rows
Progress: 70,000 rows processed, 69,576 sent, 0 errors
Progress: 80,000 rows processed, 79,582 sent, 0 errors
File 6/144: order-events+10+0000013728.snappy.parquet - 13,728 rows
Progress: 90,000 rows processed, 89,543 

In [ ]:
# Cell 7: Verify data in Kafka (optional)
# You can check AKHQ at http://localhost:8080 to see the messages in order-events-replay topic

print("Verification steps:")
print("1. Open AKHQ at http://localhost:8080")
print(f"2. Navigate to topic: {TARGET_TOPIC}")
print("3. Check that messages have been received")
print("\nOr use kafka-console-consumer:")
print(f"docker exec -it broker-i1 kafka-console-consumer --bootstrap-server localhost:9092 --topic {TARGET_TOPIC} --from-beginning --max-messages 5")